In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path().cwd() / "Data-LI"
LI_ACCOUNT_DATA_PATH = DATA_DIR / "LI-Small_accounts.csv"
LI_TRANSACTIONS_DATA_PATH = DATA_DIR / "LI-Small_Trans.csv"

accounts_df = pd.read_csv(LI_ACCOUNT_DATA_PATH)
trans_df = pd.read_csv(LI_TRANSACTIONS_DATA_PATH)

In [2]:
trans_df['Timestamp'] = pd.to_datetime(trans_df['Timestamp'])
trans_df = trans_df.rename(columns={'Account':'From Account', 'Account.1':'To Account'})
accounts_df = accounts_df.rename(columns={'Account Number': 'Account'})

accounts_df[
    'Universal_Account_ID'
    ] = accounts_df['Bank ID'].astype(str) + "_" + accounts_df['Account'].astype(str)
trans_df[
    'From_Universal_ID'
    ] = trans_df['From Bank'].astype(str) + "_" + trans_df['From Account'].astype(str)
trans_df[
    'To_Universal_ID'
         ] = trans_df['To Bank'].astype(str) + "_" + trans_df['To Account'].astype(str)

In [3]:
trans_df = trans_df.sort_values(by='Timestamp').reset_index(drop=True)

windows = ['1h', '3h', '6h', '12h', '24h']

temp_df = trans_df.set_index('Timestamp')

grouped_from = temp_df.groupby('From_Universal_ID')['Amount Paid']
grouped_to = temp_df.groupby('To_Universal_ID')['Amount Paid']
for window in windows:    
    roll_from = grouped_from.rolling(window).agg(['sum', 'count']).reset_index()
    
    roll_from = roll_from.rename(columns={
        'sum': f'from_vol_{window}',
        'count': f'from_count_{window}'
    })
    
    roll_from = roll_from.drop_duplicates(subset=['From_Universal_ID', 'Timestamp'], keep='last')
    trans_df = trans_df.merge(roll_from, on=['From_Universal_ID', 'Timestamp'], how='left')
    
    roll_to = grouped_to.rolling(window).agg(['sum', 'count']).reset_index()
    
    roll_to = roll_to.rename(columns={
        'sum': f'to_vol_{window}',
        'count': f'to_count_{window}'
    })
    
    roll_to = roll_to.drop_duplicates(subset=['To_Universal_ID', 'Timestamp'], keep='last')
    trans_df = trans_df.merge(roll_to, on=['To_Universal_ID', 'Timestamp'], how='left')

In [4]:
import numpy as np

vol_columns = [col for col in trans_df.columns if 'vol' in col]

for col in vol_columns:
    trans_df[f'{col}_log'] = np.log1p(trans_df[col])

from_log_cols = [col for col in trans_df.columns if 'from_vol' in col and 'log' in col]
to_log_cols = [col for col in trans_df.columns if 'to_vol' in col and 'log' in col]

for col in from_log_cols:
    trans_df[f'{col}_diff'] = trans_df.groupby('From_Universal_ID')[col].diff().fillna(0)
    
for col in to_log_cols:
    trans_df[f'{col}_diff'] = trans_df.groupby('To_Universal_ID')[col].diff().fillna(0)

In [5]:
trans_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 53 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Timestamp              datetime64[us]
 1   From Bank              int64         
 2   From Account           str           
 3   To Bank                int64         
 4   To Account             str           
 5   Amount Received        float64       
 6   Receiving Currency     str           
 7   Amount Paid            float64       
 8   Payment Currency       str           
 9   Payment Format         str           
 10  Is Laundering          int64         
 11  From_Universal_ID      str           
 12  To_Universal_ID        str           
 13  from_vol_1h            float64       
 14  from_count_1h          float64       
 15  to_vol_1h              float64       
 16  to_count_1h            float64       
 17  from_vol_3h            float64       
 18  from_count_3h          float64   

In [6]:

cols_to_clean = [col for col in trans_df.columns if 'Entity Name' in col or 'Universal_Account_ID' in col]
trans_df = trans_df.drop(columns=cols_to_clean, errors='ignore')

trans_df_ent = trans_df.merge(
    accounts_df[['Universal_Account_ID', 'Entity Name']], 
    left_on='From_Universal_ID', 
    right_on='Universal_Account_ID', 
    how='left'
)

trans_df_ent = trans_df_ent.drop(columns=['Universal_Account_ID'])

print(trans_df_ent.info())

<class 'pandas.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 54 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Timestamp              datetime64[us]
 1   From Bank              int64         
 2   From Account           str           
 3   To Bank                int64         
 4   To Account             str           
 5   Amount Received        float64       
 6   Receiving Currency     str           
 7   Amount Paid            float64       
 8   Payment Currency       str           
 9   Payment Format         str           
 10  Is Laundering          int64         
 11  From_Universal_ID      str           
 12  To_Universal_ID        str           
 13  from_vol_1h            float64       
 14  from_count_1h          float64       
 15  to_vol_1h              float64       
 16  to_count_1h            float64       
 17  from_vol_3h            float64       
 18  from_count_3h          float64   

In [7]:
trans_df_ent['Entity_Type'] = trans_df_ent['Entity Name'].str.replace(r'\s*#\d+', '', regex=True)

trans_df_ent = trans_df_ent.drop(columns=['Entity Name'])

print(trans_df_ent['Entity_Type'].value_counts(normalize=True))

Entity_Type
Partnership            0.358747
Sole Proprietorship    0.343523
Corporation            0.295816
Individual             0.001913
Name: proportion, dtype: float64


In [8]:
import category_encoders as ce

count_enc = ce.CountEncoder(cols=['Entity_Type'], normalize=True)

trans_df_ent['Entity_Frequency'] = count_enc.fit_transform(trans_df_ent['Entity_Type'])

trans_df_ent = trans_df_ent.drop(columns=['Entity_Type'])

print("Distribuição das Frequências (Contexto Injetado):")
print(trans_df_ent['Entity_Frequency'].value_counts().head())

Distribuição das Frequências (Contexto Injetado):
Entity_Frequency
0.358747    2483985
0.343523    2378573
0.295816    2048242
0.001913      13249
Name: count, dtype: int64


In [9]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

y_true = trans_df['Is Laundering']

features_to_keep = [
    col for col in trans_df_ent.columns 
    if ('diff' in col or 'count' in col) and 'Account' not in col
]

X = trans_df[features_to_keep]

X = X.fillna(0)

iso_forest = IsolationForest(
    n_estimators=100, 
    contamination=0.0005, 
    random_state=42, 
    n_jobs=-1
)

predictions = iso_forest.fit_predict(X)

y_pred = [1 if x == -1 else 0 for x in predictions]

print("Matriz de Confusão:")
print(confusion_matrix(y_true, y_pred))

print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=['Normal (0)', 'Lavagem (1)']))

Matriz de Confusão:
[[6917219    3265]
 [   3552      13]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00   6920484
 Lavagem (1)       0.00      0.00      0.00      3565

    accuracy                           1.00   6924049
   macro avg       0.50      0.50      0.50   6924049
weighted avg       1.00      1.00      1.00   6924049



In [10]:
features_to_keep = [
    col for col in trans_df_ent.columns 
    if ('diff' in col or 'count' in col) and 'Account' not in col
]

features_to_keep.append('Entity_Frequency')

features_to_keep

['from_count_1h',
 'to_count_1h',
 'from_count_3h',
 'to_count_3h',
 'from_count_6h',
 'to_count_6h',
 'from_count_12h',
 'to_count_12h',
 'from_count_24h',
 'to_count_24h',
 'from_vol_1h_log_diff',
 'from_vol_3h_log_diff',
 'from_vol_6h_log_diff',
 'from_vol_12h_log_diff',
 'from_vol_24h_log_diff',
 'to_vol_1h_log_diff',
 'to_vol_3h_log_diff',
 'to_vol_6h_log_diff',
 'to_vol_12h_log_diff',
 'to_vol_24h_log_diff',
 'Entity_Frequency']

In [11]:
X = trans_df_ent[features_to_keep]
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 21 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   from_count_1h          float64
 1   to_count_1h            float64
 2   from_count_3h          float64
 3   to_count_3h            float64
 4   from_count_6h          float64
 5   to_count_6h            float64
 6   from_count_12h         float64
 7   to_count_12h           float64
 8   from_count_24h         float64
 9   to_count_24h           float64
 10  from_vol_1h_log_diff   float64
 11  from_vol_3h_log_diff   float64
 12  from_vol_6h_log_diff   float64
 13  from_vol_12h_log_diff  float64
 14  from_vol_24h_log_diff  float64
 15  to_vol_1h_log_diff     float64
 16  to_vol_3h_log_diff     float64
 17  to_vol_6h_log_diff     float64
 18  to_vol_12h_log_diff    float64
 19  to_vol_24h_log_diff    float64
 20  Entity_Frequency       float64
dtypes: float64(21)
memory usage: 1.1 GB


In [12]:
iso_forest = IsolationForest(
    n_estimators=100, 
    contamination=0.0005, 
    random_state=42, 
    n_jobs=-1
)

predictions = iso_forest.fit_predict(X)
y_pred = [1 if x == -1 else 0 for x in predictions]

print("Matriz de Confusão:")
print(confusion_matrix(y_true, y_pred))

print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=['Normal (0)', 'Lavagem (1)']))

Matriz de Confusão:
[[6917036    3448]
 [   3552      13]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00   6920484
 Lavagem (1)       0.00      0.00      0.00      3565

    accuracy                           1.00   6924049
   macro avg       0.50      0.50      0.50   6924049
weighted avg       1.00      1.00      1.00   6924049

